Score commercial auto policies for expected claim risk
Fits a model on historical data and scores the current period

Data:
train.csv  - policies 2020-2022, includes claim outcomes
score.csv  - policies 2023-2024, no outcomes (to be scored)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
# ---- load ----
df2 = pd.read_csv("train.csv")
df3 = pd.read_csv("score.csv")

print("train:", df2.shape)
print("score:", df3.shape)

In [ ]:
missing = (
    df2.isna()
      .sum()
      .sort_values(ascending=False)
)

print(missing[missing > 0])

In [ ]:
missing_pct = (df2.isna().mean() * 100).sort_values(ascending=False)

print(missing_pct[missing_pct > 0])

In [ ]:
print("Duplicate rows:", df2.duplicated().sum())
print("Duplicate policy IDs:", df2["policy_id"].duplicated().sum())

In [ ]:
df2 = df2.drop_duplicates()

In [ ]:
dupes = df2[df2["policy_id"].duplicated(keep=False)].sort_values("policy_id")

for policy_id, group in dupes.groupby("policy_id"):
    for col in df2.columns:
        if group[col].nunique(dropna=False) > 1:
            print(policy_id, "->", col, ":", group[col].tolist())

In [ ]:
df2["business_type"] = df2["business_type"].str.strip().str.title()

In [ ]:
dupes = df2[df2["policy_id"].duplicated(keep=False)]

print(dupes.duplicated().sum())

In [ ]:
df2 = df2.drop_duplicates().copy()

print("Duplicate rows:", df2.duplicated().sum())
print("Duplicate policy IDs:", df2["policy_id"].duplicated().sum())

In [ ]:
df2.describe().T

In [ ]:
numeric_cols = [
    "vehicle_count",
    "vehicle_avg_age",
    "driver_count",
    "driver_avg_age",
    "years_in_business",
    "prior_year_mileage_000",
    "prior_apd_claim_count",
    "prior_al_claim_count",
    "prior_loss_amount",
    "deductible",
    "coverage_limit_000",
    "annual_premium",
    "risk_score_external",
    "num_heavy_vehicles",
    "late_payment_count"
]

for col in numeric_cols:
    print(col, "negative:", (df2[col] < 0).sum())

In [ ]:
df2[df2["annual_premium"] < 0][
    ["policy_id", "annual_premium", "coverage_type", "snapshot_date"]
]

In [ ]:
df2.loc[df2["annual_premium"] < 0, "annual_premium"] = np.nan

In [ ]:
categorical_cols = [
    "coverage_type",
    "business_type",
    "state",
    "payment_frequency",
    "claim_status_current_period"
]

for col in categorical_cols:
    print("\n", col)
    print(df2[col].value_counts(dropna=False))

In [ ]:
print("Claim rate:", (df2["claim_count"] > 0).mean())
display(df2["claim_count"].describe())
display(df2["total_loss_amount"].describe())

In [ ]:
display(
    df2.groupby("coverage_type").agg(
        policies=("policy_id", "count"),
        claim_rate=("claim_count", lambda x: (x > 0).mean()),
        avg_claim_count=("claim_count", "mean"),
        avg_loss=("total_loss_amount", "mean")
    )
)

In [ ]:
df2.groupby("business_type").agg(
    policies=("policy_id", "count"),
    claim_rate=("claim_count", lambda x: (x > 0).mean()),
    avg_loss=("total_loss_amount", "mean")
).sort_values("claim_rate", ascending=False)

In [ ]:
df2[df2["total_loss_amount"] > 0].groupby(
    "coverage_type"
)["total_loss_amount"].describe()

In [ ]:
df2[df2["total_loss_amount"] > 0].groupby(
    "business_type"
)["total_loss_amount"].describe()

In [ ]:
# ---- preprocessing ----

# fill numeric missing with 0 -- good enough for now
df2[df2.select_dtypes("number").columns] = df2.select_dtypes("number").fillna(0)
df3[df3.select_dtypes("number").columns] = df3.select_dtypes("number").fillna(0)

# encode categoricals for train
df2["coverage_type"] = df2["coverage_type"].astype("category").cat.codes
df2["business_type"]  = df2["business_type"].astype("category").cat.codes
df2["state"]          = df2["state"].astype("category").cat.codes

# same for score
df3["coverage_type"] = df3["coverage_type"].astype("category").cat.codes
df3["business_type"]  = df3["business_type"].astype("category").cat.codes
df3["state"]          = df3["state"].astype("category").cat.codes

In [ ]:
# ---- preprocessing ----

# fill numeric missing with 0 -- good enough for now
df2[df2.select_dtypes("number").columns] = df2.select_dtypes("number").fillna(0)
df3[df3.select_dtypes("number").columns] = df3.select_dtypes("number").fillna(0)

# encode categoricals for train
df2["coverage_type"] = df2["coverage_type"].astype("category").cat.codes
df2["business_type"]  = df2["business_type"].astype("category").cat.codes
df2["state"]          = df2["state"].astype("category").cat.codes

# same for score
df3["coverage_type"] = df3["coverage_type"].astype("category").cat.codes
df3["business_type"]  = df3["business_type"].astype("category").cat.codes
df3["state"]          = df3["state"].astype("category").cat.codes

In [ ]:
features = [
    "vehicle_count",
    "vehicle_avg_age",
    "driver_count",
    "driver_avg_age",
    "years_in_business",
    "prior_year_mileage_000",
    "business_type",
    "state",
    "prior_apd_claim_count",
    "prior_al_claim_count",
    "prior_loss_amount",
    "deductible",
    "coverage_limit_000",
    "annual_premium",
    "risk_score_external",
    "has_safety_program",
    "num_heavy_vehicles",
    "coverage_type"
]

In [ ]:
df2["has_claim"] = (df2["claim_count"] > 0).astype(int)

In [ ]:
df2["snapshot_date"] = pd.to_datetime(df2["snapshot_date"])

train_data = df2[df2["snapshot_date"].dt.year <= 2021]
valid_data = df2[df2["snapshot_date"].dt.year == 2022]

In [ ]:
X_train = train_data[features]
y_train = train_data["has_claim"]

X_valid = valid_data[features]
y_valid = valid_data["has_claim"]

In [ ]:
model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

In [ ]:
pred_prob = model.predict_proba(X_valid)[:, 1]

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_valid, pred_prob)

print("ROC-AUC:", auc)

In [ ]:
from sklearn.metrics import average_precision_score

ap = average_precision_score(y_valid, pred_prob)

print("Average Precision:", ap)